In [9]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_community.utilities.google_serper import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langchain_groq import ChatGroq

from langgraph.checkpoint.memory import InMemorySaver


search = GoogleSerperAPIWrapper()


@tool
def web_search(query: str) -> str:
    """Search Google and return relevant search results."""
    return search.run(query)


# llm = init_chat_model(
#     model="gemini-3.5-flash-lite",
#     model_provider="google_genai",
# )

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0.3,
)


agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
You are a research assistant.

Use Google Search whenever the question requires current
or factual information.

Answer the user's question based on the search results.
If something cannot be verified, clearly say so.

Give a concise, direct answer.
""",
checkpointer=InMemorySaver(),
)


thread_config = {
    "configurable": {
        "thread_id": "langgraph-inmemory-demo",
    }
}


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "EWC PUBG 2026 final match at what time and where?",
            }
        ]
    },
    config=thread_config,
)


last_message = response["messages"][-1]
content = last_message.content

if isinstance(content, list):
    print(content[0].get("text", content[0]))
else:
    print(content)